Author: Robin Sternberg  
This notebook is licensed under Creative Commons Attribution-ShareAlike 4.0

# Text Parsing

Dieses Notebook extrahiert und verarbeitet Daten aus Protokollen des Deutschen Bundestages mithilfe von regulären Ausdrücken, um Namen, Unterbrechungen, Applaus und Redebeiträge zu identifizieren und zu parsen.

In [2]:
import json
import re
import ntpath
import os
import pandas as pd

# 1. Regular Expressions

In [3]:
#parties = ['CDU/CSU', 'GRÜNE','SPD', 'FDP', 'AfD', 'DIE LINKE', 'KPD', 'BP', 'DP', 'WAV', 'Z', 'fraktionslos' ]    # includes historic parties
#partei_match = r'(CDU\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|KPD|BP|DP|WAV|Z|DIE LINKE|Die Linke|fraktionslos)'            # not used because of limiting to protocols since 1991

parties = ['CDU/CSU', 'GRÜNE','SPD', 'FDP', 'AfD', 'DIE LINKE','fraktionslos'] 
partei_match = r'(CDU\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|DIE LINKE|fraktionslos)'

### Namen
Um Namen zu erkennen, mussten wir alle Eventualitäten und Konventionen der Protokollanten abbilden. Außerdem gibt es ein paar echt interessante Einzelfälle...

In [4]:
name_match = ( '('
              +  r'(?:Dr\.\s)?'   # Optional "Dr. "
              + r'(?:\w+(?:-\w+)?\s)'   # First name (or hyphenated first name)
              + r'(?:\w+\.\s)?'   # Optional middle initial
              + r'(?:von\s)?'      # Optional Title
              + r'\w+(?:-\w+)?'   # Last name (or hyphenated last name)
              + ')'
             )

In [5]:
print(re.findall(name_match, 'Dr. Harald J. Töpfer'))
print(re.findall(name_match, 'Dr. Marie-Agnes Strack-Zimmermann'))
print(re.findall(name_match, 'Rhaenyra von Drachenstein'))
print(re.findall(name_match, 'Brynden B. Tully'))
print(re.findall(name_match, 'Oberyn Martell'))

['Dr. Harald J. Töpfer']
['Dr.\xa0Marie-Agnes Strack-Zimmermann']
['Rhaenyra von Drachenstein']
['Brynden B. Tully']
['Oberyn Martell']


### Unterbrechungen

Unterbrechungen sind Einwürfe aus den Reihen des Bundestags, bei denen der Sprecher identifizierbar war und deswegen mit vermerkt wurde. 

In [6]:
unterbrechung_match = re.compile(rf'{name_match}\s+' # interruptions are comprised of a name
                                rf'\[{partei_match}\]:\s' # ...followed by the party in square brackets
                                r'([\s\S]*?)' # ...followed by the comment text including newlines
                                r'(?:[-–—\)]\s)', # ...ended by some sort of hyphen or closing round brackets followed by some whitespace char
                                flags = re.UNICODE)

In [7]:
unterbrechungen_beispiel = """...mit Ihren Ankündigungen zu höheren Lebensmittelpreisen erreichen Sie nur eines: Sie sorgen für Verunsicherung.
(Beifall bei der CDU/CSU – Renate Künast [GRÜNE]: Wie? Das hat doch Klöckner auch getan! – Harald Ebner [GRÜNE]: Das ist doch peinlich!)
... Hier steht weiterer Redetext (Bernd Höfer [AfD]: Das hier ist ein Zuruf mit Binde-Strich in der Mitte! - Robert Baratheon [CDU/CSU]: Diese Unterbrechung ist direkt nach der Vorigen und nur durch '-' getrennt) 
"""
re.findall(unterbrechung_match, unterbrechungen_beispiel)

[('Renate Künast', 'GRÜNE', 'Wie? Das hat doch Klöckner auch getan!\xa0'),
 ('Harald Ebner', 'GRÜNE', 'Das ist doch peinlich!'),
 ('Bernd Höfer',
  'AfD',
  'Das hier ist ein Zuruf mit Binde-Strich in der Mitte! '),
 ('Robert Baratheon',
  'CDU/CSU',
  "Diese Unterbrechung ist direkt nach der Vorigen und nur durch '-' getrennt")]

In [8]:
test_text = """
Berlin gesagt hat. Seine klare Aussage lautete: Wenn die
Nachbarländer am Jahresende 2002 aus der
Subventionierung des Diesels nicht ausstiegen, würden wir diese
Subventionierung einführen. Wenn Sie sich an diesen
Worten messen lassen, dann haben Sie uns auf Ihrer Seite. 
(Beifall bei der FDP und der CDU/CSU – 
Dr. Margit Spielmann [SPD]: Das war die
neueste Nachricht! Die musste heute noch sein!)
Vizepräsident Dr. Hermann Otto Solms: 
Wollen Sie erwidern? – Bitte schön, Herr Schmidt.
Albert Schmidt (Ingolstadt) (GRÜNE): 
Ich mache es kurz. Ich kann nur wiederholen, was ich
vorhin gesagt habe; denn genauso habe ich das gemeint.
"""

re.findall(unterbrechung_match, test_text)

[('Dr. Margit Spielmann',
  'SPD',
  'Das war die\nneueste Nachricht! Die musste heute noch sein!')]

In [9]:
print(unterbrechung_match)

re.compile('((?:Dr\\.\\s)?(?:\\w+(?:-\\w+)?\\s)(?:\\w+\\.\\s)?(?:von\\s)?\\w+(?:-\\w+)?)\\s+\\[(CDU\\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|KPD|BP|DP|WAV|Z|DIE LINKE|fraktionslos)\\]:\\s([\\s\\S]*?)(?:[-–—\\)]\\s)')


### Zurufe

Zurufe sind Unterbrechungen des Redners, bei denen der genaue Unterbrecher nicht mitprotokolliert wurde. Diese zu finden ist auch nicht sonderlich schwierig.

In [10]:
zuruf_match = re.compile(r'[-–—\s\(]'   # Interruptions with unknown speakers but known party affiliation are comprised of a leading hyphen or opening parenthesis
                         r'Zuruf .*?'   # ... followed by 'Zuruf' and some more irrelevant fillwords
                         rf'{partei_match}:\s'  # ... folllowed by the party name and :
                         r'([\s\S]*?)'     # ... followed by the text of the comment
                         r'[-–—\)]\s',    # ... ended by some sort of hyphen or closing round parenthesis followed by some whitespace char
                        flags = re.UNICODE)

In [11]:
zuruf_beispiel = """Danach, Frau Kollegin Matthäus-Maier, wurde es 
leider wieder langweilig, enttäuschend: das gleiche 
Ritual, 
(Zuruf von der CDU/CSU: Immer das
selbe!) 
leider auch immer wieder Unterstellungen. Bei Ihnen 
taucht sehr oft das Wort „ehrlich" auf. 
(Zuruf von der CDU/CSU: Das ist immer verdächtig!) 
Sie sollten es allerdings nicht permanent mit
Unterstellungen
"""

re.findall(zuruf_match, zuruf_beispiel)

[('CDU/CSU', 'Immer das\nselbe!'), ('CDU/CSU', 'Das ist immer verdächtig!')]

### Beifall

Interessanterweise wird in den Protokollen auch Beifall vermerkt. Diese Daten lassen wir natürlich nicht unangetastet.

In [12]:
beifall_match = re.compile(r'[-–—\s\(]' # Applause is comprised of a leading hyphen or opening parenthesis
                           'Beifall'    # ...followed by "Beifall"
                           r'[\s\S]*?'  # ...followed by a sentence including the party names
                           r'[-–—\)]',  # ...ended by some sort of hyphen or closing round parenthesis
                           flags = re.UNICODE)

In [13]:
beifall_beispiel = """(Beifall bei der CDU/CSU sowie bei
Abgeordneten der SPD, des GRÜNE und der FDP – Stephan Brandner 
[AfD]: Erbärmliche Wendehälse!)"""
re.findall(beifall_match, beifall_beispiel)


['(Beifall bei der CDU/CSU sowie bei\nAbgeordneten der SPD, des GRÜNE und der FDP –']

Der gematchte String wird dann später weiterverarbeitet. Um die Beifall gebenden Parteien zu finden, sieht man einfach nach, ob deren Kürzel im String vorkommt.  
Man sieht hier auch die Ersetzung von "Bündnisses 90/die Grünen" in der load_text() Methode, was wie hier zu fehlerhafter Grammatik führen kann.

### Redebeiträge/Sprecher

Damit zu jeder Unterbrechung auch vermerkt werden kann, wer denn überhaupt unterbrochen wurde, muss das natürlich auch erkannt werden. Das war aber leichter gesagt, als getan.

speaker_match ist ein RegEx, der immer nur Stück für Stück erweitert werden konnte, als wieder eine neue Eventualität nicht vom vorigen RegEx erfasst wurde.
Im Git finden sich bestimmt 10 unterschiedliche Versionen. Es kann auch sein, dass mit dieser Variante ein paar Sonderfälle nicht zu 100% abgedeckt sind.  
Für kleinere RegEx konnte der Chatbot unseres Vertrauens noch sinnvolle Ausgaben machen (z.B. beim Namen gingen Teile davon noch ganz gut).
Die Aufgabe hier war aber deutlich zu

Es gibt mehrere Varianten, wie eine Rede gestartet werden kann. 
* Variante 1 - die leicht Erkennbare:  
  * Ein Name (name_match) gefolgt von runden Klammern, in denen das Parteikürzel steht  
  * **Beispiele:** 
    * >Ingo Gädechens (CDU/CSU):  
      >Frau Präsidentin! Liebe Kolleginnen und Kollegen! ...
    * >René Springer (AfD):  
      >Vielen Dank, Frau Präsidentin...

* Variante 2 - die schwierig Erkennbare:  
  * Staatssekretäre, Bundesminister, Bundeskanzler etc. werden mit ihrem Titel und ohne Angabe der Partei vermerkt.  
  Das bedeutet, dass...  
    1. man die Parteizugehörigkeit (soweit vorhanden) nachschlagen muss.
    2. die Erkennung deutlich schwieriger wird, weil die runden Klammern mit Parteizugehörigkeit nicht mehr da sind, um den RegEx einzuschränken.  
  Was bleibt ist Trial and Error, um letztendlich herauszufinden, dass zum Glück alle dieser Redeanfänge nur einen der folgenden Anfänge nach dem Komma haben  
    (Parl\. Staatssekretär|Staatssekretär|Bundeskanzler|Bundesminister)  
  Außerdem findet man erst heraus, dass z.B. .*? sich nicht zum Matchen der unbekannten Anteile, wie "beim Bundesminster des Inneren" eignet, wenn man den RegEx auch auf Protokolle aus dem 90ern anwendet, da hier viel öfter Newlines vorhanden sind.  
  Nach jeder Änderung muss man die RegEx auf 4 unterschiedlichen Protokollen im Zeitraum 2024 bis 1991 ausprobieren, um zu sehen, ob nicht Dinge übersehen werden.  
  Die jetzige Version lässt vom Namen bis zum ':' maximal eine Newline zu und ist deutlich simpler, als so manche Zwischenschritte auf dem Weg dahin.
  * **Beispiele** (mit akkurat gesetzten Newlines):  
    * >Manfred Kanther, Bundesminister des Innern: Frau Präsidentin!... 
    * >Fritz Rudolf Körper, Parl. Staatssekretär beim  
      >Bundesminister des Innern:  
      >Herr Kollege Grindel...
    * >Christian Lindner, Bundesminister der Finanzen:  
      >Sehr geehrte, liebe Frau Kollegin ...
  * **Antibeispiel** (mit akkurat gesetzten Newlines):  
    * >\n  
      >Arbeitsvertrag unterschrieben, und urplötzlich fällt Herrn Staatssekretär Graichen ein: Oh, das ist ja mein Trauzeuge.  
      
      Das hier hat z.B. in einer vorigen Variante gematcht. "Arbeitsvertrag unterschrieben" war der Name, "Staatssekretär" ist in den nächsten 100 Zeichen vorgekommen und ein richtig gesetzets "," und ":" gab es auch noch.  




In [14]:
speaker_match = re.compile(
    rf'{name_match}\s(?:\(\w+\)\s)?\({partei_match}\):'  # Matches speaker's name, optional city name, followed by party name in parentheses, e.g., "Harald Töpfer (Party):"
    '|'
    r'\n'  # Newline
    rf'{name_match}, '
    r'((?:Parl\. Staatssekretär|Staatssekretär|Bundeskanzler|Bundesminister))[\w ]{0,100}\n?[\w ]{0,100}?:',  # Matches name followed by official title and fillwords with no more than one newline, e.g. "Harald Töpfer, Bundeskanzler:" "Ronald Wiesel, Staatssekretär {...}:"
    flags=re.UNICODE
)
# bottom part of the regex (after |) has two capture-groups too so that the alignment of names is not out of order when splitting
# conveniently, no distinction between male and female Staatsminister(in) , Saatssekretär(in), Bundeskanzler(in) etc. has to be made as they start with the same chars
speaker_match

re.compile(r'((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)(?:\w+\.\s)?(?:von\s)?\w+(?:-\w+)?)\s(?:\(\w+\)\s)?\((CDU\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|KPD|BP|DP|WAV|Z|DIE LINKE|fraktionslos)\):|\n((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)(?:\w+\.\s)?(?:von\s)?\w+(?:-\w+)?), ((?:Parl\. Staatssekretär|Staatssekretär|Bundeskanzler|Bundesminister))[\w ]{0,100}\n?[\w ]{0,100}?:',
           re.UNICODE)

Ziemlich lang und unübersichtlich...

In [15]:
speaker_strack_zimmermann = 'Dr. Marie-Agnes Strack-Zimmermann (FDP): '
print(re.findall(speaker_match, speaker_strack_zimmermann ))

speaker_farle = '''Robert Farle
(fraktionslos):
Sehr geehrter Herr Präsident! Sehr'''
print(re.findall(speaker_match, speaker_farle))

speaker_koerper = '''
Rudolf Körper, Parl. Staatssekretär beim  
Bundesminister des Innern:  
Herr Kollege Grindel ...'''
print(re.findall(speaker_match, speaker_koerper))

speaker_scholz = '''
Olaf Scholz, Bundeskanzler:
Es ist sehr gut'''
print(re.findall(speaker_match, speaker_scholz ))

[('Dr. Marie-Agnes Strack-Zimmermann', 'FDP', '', '')]
[('Robert Farle', 'fraktionslos', '', '')]
[('', '', 'Rudolf Körper', 'Parl. Staatssekretär')]
[('', '', 'Olaf Scholz', 'Bundeskanzler')]


#### **Formate** 
Schwierig wurde die Entwicklung der RegEx vor allem dadurch, dass sich das Format in den Jahren bis 1991 mehrfach geändert hat.  
Die heutigen Protokolle sind meistens schön formatiert, haben mehrere Newlines vor einem Redeanfang und sind für die Sicht am Computer gemacht. 
Die alten Protokolle wurden aber wie [hier](https://dserver.bundestag.de/btp/12/12093.pdf) in zwei Spalten gedruckt. Das führt zu sehr vielen Zeilenumbrüchen mitten in Sätzen. 

Der Grund, warum unsere Analysen nur bis 1991 zurückreichen ist, dass frühere Protokolle in einem gravierend anderen Format geschrieben sind. Hier wird immer nur der Nachname der Abgeordneten genannt. Um zusätzlich Namenskonflikte zu beheben wird dahinter in runden Klammern auch die Stadt/der Wahlkreis/der Geburtsort? angegeben. Wir haben ohnehin schon sehr viele Daten, weshalb ich mich lieber besseren Analysen gewidmet habe, als noch mehr RegEx zu schreiben.
Anmerkung: Das Format hat sich von 1949 bis 1991 noch mehrere Male geändert. Ein einzelner Extra-Satz RegEx hätte also nicht gereicht, um die gesamte Geschichte des Bundestags abzubilden.

### Fehler in den Protokollen 

Die Entwicklung der RegEx wurde auch noch durch teils gravierende Fehler in den Protokollen schwer gemacht.  
Manchmal sind es nur kleine Dinge. z.B. wenn der Protokollant/die Protokollantin nur CDU anstatt CDU/CSU schreibt. Das lässt sich ja relativ leicht lösen.  

Manchmal gehen aber ganze Teile des Protokolls kaputt.

**Beispiel:** Die JSON-Datei, die z.B. für das Protokoll am 13. April 2000 von der API bereitgestellt wurde. Der innerhalb der JSON-Datei gelieferte Text ist ohne Veränderung in ./data/examples als .txt Datei zu finden.
Als zum Informatikstudium passenden Abschnitt empfehle ich Zeile 3028-3050.  
Hier wurden die rechte und linke Seite des Quell-Drucks, beim digitalen Einlesen des Ursprungs-Protokolls in Papierform, zusammengefügt.  
Resultat ist ein Text, der mittendrin zwischen den beiden Spalten hin und herspringt.
<details>
  <summary>Text</summary>

>Diesen Lernprozess begrüßen wir sehr, wir begrüßen der Bundesministerin für Bildung und Forschung: Bitte.  
>auch den Sinneswandel von Frau Merkel, die am Montag  
>dieser Woche erklärt hat, die CDU habe keine grundsätz- Dr. Christa Luft (PDS): Herr Kollege Catenhusen, Sie  
>lichen Einwände gegen diese Aktion. Sicherlich ist Ihr haben eben betont, dass es in der IT-Branche vor allem  
>Menschen mit Hochschulabschluss geben müsse. Was sa- Menschen aus guten Gründen, übrigens auch unterstützt (C)  
>gen Sie denn zu der Feststellung des SPD-Fraktionschefs, von der eigenen Regierung, auf diesen globalen  
>Arbeitsdie in der „FAZ“ vom 13. April wiedergegeben wird? Er markt drängt, dann ist die klassische Diskussion, zum  
>Beisoll gesagt haben, die SPD-Fraktion lege auch keinen all- spiel um Braindrain wie in den 70er-Jahren, hier völlig  
>zu großen Wert auf formale Hochschulabschlüsse, sondern fehl am Platze.   
>neben der Orientierung an formalen  
>Hochschulabschlüssen allein, wie sie im Entwurf von Arbeitsminister Riester (Beifall bei Abgeordneten der SPD)  
>zunächst vorgesehen sei, werde daran gedacht, sich an den Indien profitiert davon, dass es eine starke Gruppe von  
>gezahlten Gehältern für Fachleute aus dem Nicht-EU- Indern gibt, die die Computerindustrie in Silicon Valley  
>Raum zu orientieren. Diese liegen ja wohl nicht sehr hoch. mit aufgebaut haben.  
>Wolf-Michael Catenhusen, Parl. Staatssekretär bei (Jörg Tauss [SPD]: Heute schon!)  
>der Bundesministerin für Bildung und Forschung: Der Wir werden auf die Dauer davon profitieren, dass in unse-  
>Teufel liegt immer im Detail, Frau Luft. ren Unternehmen hoch qualifizierte Spezialisten arbeiten,  
>Übrigens ist meine Prognose ganz klar: Es werden nicht die anschließend in ihren eigenen Ländern unsere  
>Gehauptsächlich Inder sein, sondern zu uns werden vor allem schäftspartner werden.   
>hoch qualifizierte Experten aus dem osteuropäischen (Beifall bei der SPD sowie des Abg. Matthias  
>Raum kommen. Das weiß doch jeder. Berninger [BÜNDNIS 90/DIE GRÜNEN])  
</details>

Das macht eine Auswertung natürlich zwangsweise fehlerbehaftet. Weil es aber schlicht unmöglich ist, alles probezulesen müssen wir über solche Fehler aber leider hinwegsehen und nehmen alle Protokolle gleich auf.  
Dem Problem könnte sich vielleicht ein zukünftiges Team annehmen, das analysiert, wie oft solche Fehler vorkommen, und dann ggf. den zuständigen Stellen Bescheid gibt.

# 2. Laden von Stammdaten aller Bundestagsabgeordneten 

Hier laden wir zum Nachschlagen von Sprechern ohne gegebene Parteizugehörigkeit (Staatssekretäre, Bundesminister etc.) eine Liste an Namen und Parteien, die im Notebook "get_abgeordneten_data" erstellt wurde.

In [16]:
import csv
bt_tuples = []
with open('../_data/abgeordnete.csv', 'r', newline= '',encoding='utf8') as file:
    reader = csv.reader(file)
    next(reader) # skip first row
    for row in reader:
        bt_tuples.append(tuple(row))


Manche Sprecher werden trotzdem nicht gefunden, weil sie nie Teil des Bundestags waren. Das trifft auf manche Staatssekretäre(innen) und Minister(innen) zu.
Fälle, die besonders oft vorkamen, wurden händisch in get_abgeordneten_data hinzugefügt.

In [17]:
def find_party_by_name(name, tuple_list):
    tuple_list_sorted = sorted(tuple_list, key = lambda x: x[2], reverse = True) # we focus on 
    if name.startswith('Dr.'):
        name = name[4:]
    for item in tuple_list_sorted:
        if name in item[0]:
            if item[1] != 'CSU' and item[1] != 'CDU':
                return item[1]
            else:
                return 'CDU/CSU'  
    return "<unknown>"

print(find_party_by_name('Dr. Robert Habeck', bt_tuples))
print(find_party_by_name('Winfried Fockenberg', bt_tuples))
print(find_party_by_name('Olaf Scholz', bt_tuples))

GRÜNE
CDU/CSU
SPD


Das absteigende Sortieren nach Wahlperioden in der Methode hilft, Fehler zu vermeiden. Wir schauen uns (zumindest derzeit) hauptsächlich neue Daten an.  
Deshalb reicht es aus, die wenigen Namenskonflikte in der Liste zu behandeln, indem wir einfach den neueren Eintrag nehmen.

# 3. Parsen eines Protokolls

### Laden des Rohtexts aus den über die API bereitgestellten JSON-Dateien

Hier werden schon erste Vorverarbeitungsschritte angewendet, um z.B. unterschiedliche Namen für die gleiche Partei auf den gleichen Nenner zu bringen.

In [38]:
def load_text(path_to_json):
    with open(path_to_json) as f:
        dictionary = json.load(f)
        if 'text' in dictionary:
            text = dictionary['text']
        else:
            return 'no text found'
    text = re.sub(r'\n-\n', '-', text) # weird formatting probably due to digitalization from printed mediums
    text = re.sub(r'DIE\s*?LINKE|PDS/\s*?Linke\s*?Liste|PDS/LL|PDS|Die\s*?Linke', 'DIE LINKE', text) # fusion of PDS and linke Liste and other spelling
    text = re.sub(r'F\.D\.P\.', 'FDP', text) # old style of writing FDP
    buendnis_match = re.compile(r'(BÜND(-\n?)?NIS((-\n?)?SES)?\s*?90\/\s*(DIE\s*?)?GRÜ(-\n?)?NEN?)', flags=re.UNICODE|re.IGNORECASE) # hyphens, newlines, plural etc.etc.etc.
    text = re.sub(buendnis_match,'GRÜNE', text) 
    return text

In [37]:
demo_text = load_text('../_data/protocols/20_050_2022-09-07.json')
print(demo_text[:300])

Plenarprotokoll20/50

Deutscher Bundestag
Stenografischer Bericht

50. Sitzung

Berlin,Mittwoch, den 7. September 2022



Inhalt:

Gedenken an Michail Gorbatschow



Begrüßung des Direktors beim Deutschen Bundestag, Herrn Dr. Michael Schäfer



Tagesordnungspunkt 1 (Fortsetzung):

a) Erste Beratung 


### Aufteilen des Texts in Reden/Redebeiträge

Zu Redebeiträgen zählen z.B. auch Zwischenfragen (nicht zu verwechseln mit Zwischenrufen)

Der Regex speaker_match teilt das Protokoll in eine Liste von Strings auf, wobei sich das Muster alle 5 Listeneinträge wiederholt.  
1. Eintrag: Sprechername, falls Standard-Schema
2. Eintrag: Partei, falls Standard-Schema
3. Eintrag: Sprechername, falls Bundeskanzler, Bundesminister, Staatssekretär etc.
4. Eintrag: Titel, falls Bundeskanzler, Bundesminister, Staatssekretär etc. (Bemerkung: der 4. Eintrag wird im Parsing nicht genutzt, ich fand es aber trotzdem z.B. zum Debuggen sinnvoll hier noch eine Capture Group zu haben)
5. Eintrag: Text der Rede (Text bis zum nächsten speaker_match)

In [36]:
speeches_raw_demo = re.split(speaker_match, demo_text)[1:] # split and discard the document-head 
speeches_raw_demo[:10]

['Friedrich Merz',
 'CDU/CSU',
 None,
 None,
 '\nFrau Präsidentin! Liebe Kolleginnen und Kollegen! Es ist mehr als angemessen, dass wir heute Morgen des verstorbenen letzten Staatspräsidenten der ehemaligen Sowjetunion und Generalsekretärs der KPdSU gedacht haben. Wir verdanken Helmut Kohl, George Bush und eben auch Michail Gorbatschow die Chance, dass unser Land vor über 30\xa0Jahren die Wiedervereinigung in Frieden und Freiheit erreichen konnte.\n(Beifall bei der CDU/CSU sowie bei Abgeordneten der SPD, des GRÜNE, der FDP, der AfD und der LINKEN)\nHätte Gorbatschow seinen politischen Weg weitergehen können, wären Glasnost und Perestroika die prägenden Elemente der russischen Politik nach dem Ende der Sowjetunion geblieben, dann wäre nicht nur die russische Geschichte anders verlaufen. Die gesamte europäische Geschichte wäre anders verlaufen. Aber spätestens seit dem russischen Angriffskrieg gegen die Ukraine sind wir auf schreckliche Weise mit einer ganz anderen Realität konfrontiert,

In [21]:
speech_list = []

for i in range(0, len(speeches_raw_demo), 5):
        if speeches_raw_demo[i] != None and speeches_raw_demo[i+1] != None:
            name = speeches_raw_demo[i].strip()    
            party = speeches_raw_demo[i+1].strip()
            if party == 'CSU' or party == 'CDU': # sometimes the the secretary tasked with writing everything down forgets to put CDU/CSU instead of CSU or CDU
                party = 'CDU/CSU'
        else:
            name = speeches_raw_demo[i+2].replace('\n', ' ').strip() # sometimes the formatting is all over the place e.g. "Olaf\nScholz" instead of "Olaf Scholz" :)
            party = find_party_by_name(name, bt_tuples)

        speech = {
            'speaker':{
                'name':name,
                'party':party
            },
            'text': re.split(r'(\nVizepräs.{0,99}?:)|(\nPräsid.{0,99}?:)|(\nAnlage)',speeches_raw_demo[i+4])[0].strip() # handle end of File (Anlagen) and interruptions by the Bundestagspräsident(in) or Vice Bundestagspräsident(in)
        }

        if len (speech_list) >=1 and speech_list[-1]['speaker'] == speech['speaker']:
            speech_list[-1]['text'] += '\n' + speech['text'] # append speeches of the same speaker which occur right after each other. This can e.g. happen due to interruptions by the president of the Bundestag
        else:
            speech_list.append(speech)

speech_list[:3]

[{'speaker': {'name': 'Friedrich Merz', 'party': 'CDU/CSU'},
  'text': 'Frau Präsidentin! Liebe Kolleginnen und Kollegen! Es ist mehr als angemessen, dass wir heute Morgen des verstorbenen letzten Staatspräsidenten der ehemaligen Sowjetunion und Generalsekretärs der KPdSU gedacht haben. Wir verdanken Helmut Kohl, George Bush und eben auch Michail Gorbatschow die Chance, dass unser Land vor über 30\xa0Jahren die Wiedervereinigung in Frieden und Freiheit erreichen konnte.\n(Beifall bei der CDU/CSU sowie bei Abgeordneten der SPD, des GRÜNE, der FDP, der AfD und der LINKEN)\nHätte Gorbatschow seinen politischen Weg weitergehen können, wären Glasnost und Perestroika die prägenden Elemente der russischen Politik nach dem Ende der Sowjetunion geblieben, dann wäre nicht nur die russische Geschichte anders verlaufen. Die gesamte europäische Geschichte wäre anders verlaufen. Aber spätestens seit dem russischen Angriffskrieg gegen die Ukraine sind wir auf schreckliche Weise mit einer ganz anderen

### Extrahieren von Zwischenrufen/Unterbrechungen aus dem Redetext

In [22]:
for speech in speech_list:
    comments = []
    for match in re.finditer(unterbrechung_match,speech['text']):
        # comments with known speaker
        comment = {
            'commentator': {
                'name': match.group(1),
                'party': match.group(2)
            },
            'text': match.group(3).strip(),
            'preceding_context': speech['text'][:match.start()] # include all text until comment for later training of LLM by Linh
        }
        comments.append(comment)

    # comments with unknown speaker (Zurufe)
    for match in re.finditer(zuruf_match, re.sub('der LINKEN', 'DIE LINKE', speech['text'])):
        comment = {'commentator':{
                'name': '<unknown>',
                'party': match.group(1)
            },
            'text': match.group(2),
            'preceding_context': speech['text'][:match.start()]
        }
        comments.append(comment)

    speech['comments'] = comments

Die Entscheidung, für jeden Text auch den gesamten vorigen Kontext der Rede doppelt abzuspeichern, war dem geschuldet, dass ich es für meine Team-Kameraden möglichst leicht machen wollte an die Daten zu kommen.  
Ich hätte auch einfach den Startindex der Unterbrechung abspeichern können. In der jetzigen Form ist die Datenaufmachung etwas simpler und man kann, ohne darüber nachzudenken, nur die Zwischenrufe selbst extrahieren, weil in den Json-Objekten schon alles enthalten ist, was man für z.B. die Zwischenruf-Generierung benötigt.

In [44]:
for speech in speech_list[:1]:
    if len(speech['comments']) != 0:
        for comment in speech['comments']:
            print(comment)

{'commentator': {'name': 'Robert Farle', 'party': 'AfD'}, 'text': 'Das stimmt doch nicht! Das wissen Sie doch!', 'preceding_context': 'Frau Präsidentin! Liebe Kolleginnen und Kollegen! Es ist mehr als angemessen, dass wir heute Morgen des verstorbenen letzten Staatspräsidenten der ehemaligen Sowjetunion und Generalsekretärs der KPdSU gedacht haben. Wir verdanken Helmut Kohl, George Bush und eben auch Michail Gorbatschow die Chance, dass unser Land vor über 30\xa0Jahren die Wiedervereinigung in Frieden und Freiheit erreichen konnte.\n(Beifall bei der CDU/CSU sowie bei Abgeordneten der SPD, des GRÜNE, der FDP, der AfD und der LINKEN)\nHätte Gorbatschow seinen politischen Weg weitergehen können, wären Glasnost und Perestroika die prägenden Elemente der russischen Politik nach dem Ende der Sowjetunion geblieben, dann wäre nicht nur die russische Geschichte anders verlaufen. Die gesamte europäische Geschichte wäre anders verlaufen. Aber spätestens seit dem russischen Angriffskrieg gegen die

### Extrahieren von Applaus aus dem Redetext

Hier werden alle vom Regex `beifall_match` gematchten Strings zusammengefügt und dann nach den Vorkommnissen von Parteinamen durchsucht.  
Dabei wird dann an der richtigen Stelle im Dictionary hochgezählt.

In [24]:
for speech in speech_list:
    beifall = re.findall(beifall_match, speech['text'])
    beifall = ''.join(beifall)
    beifall = re.sub('der LINKEN', 'DIE LINKE', beifall)
    beifall_counts = {party: beifall.count(f' {party}') for party in parties}
    speech['applause'] = beifall_counts

Rede von Friedrich Merz...

In [25]:
speech_list[0]['applause']

{'CDU/CSU': 30,
 'GRÜNE': 2,
 'SPD': 3,
 'FDP': 3,
 'AfD': 8,
 'DIE LINKE': 4,
 'KPD': 0,
 'BP': 0,
 'DP': 0,
 'WAV': 0,
 'Z': 0,
 'fraktionslos': 0}

Rede von Olaf Scholz...

In [26]:
speech_list[1]['applause']

{'CDU/CSU': 0,
 'GRÜNE': 40,
 'SPD': 40,
 'FDP': 38,
 'AfD': 1,
 'DIE LINKE': 0,
 'KPD': 0,
 'BP': 0,
 'DP': 0,
 'WAV': 0,
 'Z': 0,
 'fraktionslos': 0}

Die eigenen Parteien bzw. Koalitionspartner unterstützen den Redner also doch recht eindeutig. Die Zahlen haben allerdings etwas Rauschen, weil der Beifall auch auf einen negativen Zuruf einer kritisch gestimmten Partei bezogen sein kann. Mehr zum Applaus gibt es in der statistischen Analyse.

## Recap: Wie sehen unsere geparsten Reden aus

Es gibt Informationen über den Sprecher...

In [27]:
speech_list[1]['speaker']

{'name': 'Olaf Scholz', 'party': 'SPD'}

... den Text der Rede

In [28]:
print(speech_list[1]['text'][:1000]) # only the first 1000 chars for non-collapsed output

Sehr geehrte Frau Präsidentin! Meine verehrten Kolleginnen und Kollegen! Verehrter Herr Kollege Merz, ich habe Ihnen eben sehr genau zugehört.
(Hermann Gröhe [CDU/CSU]: Und schon wieder alles vergessen!)
Ich will Ihnen eins antworten: Unterschätzen Sie unser Land nicht! Unterschätzen Sie nicht die Bürgerinnen und Bürger dieses Landes!
(Beifall bei der SPD, dem GRÜNE und der FDP – Lachen des Abg. Tino Chrupalla [AfD])
In schweren Zeiten wächst unser Land über sich selbst hinaus.
(Tino Chrupalla [AfD]: Haken wir uns unter!)
Wir haben eine gute Tradition, uns unterzuhaken, wenn es schwierig wird:
(Beifall bei Abgeordneten der AfD – Tino Chrupalla [AfD]: Ja! Wir haken uns unter!)
Bund, Länder und Kommunen, Politik, Zivilgesellschaft, Arbeitgeber und Betriebsräte, Unternehmen und Gewerkschaften. Wer Spaltung herbeiredet, der gefährdet den Zusammenhalt in diesem Land, und das ist jetzt das Falsche.
(Beifall bei der SPD, dem GRÜNE und der FDP – Tino Chrupalla [AfD]: Sie spalten! – Thorsten Fr

... die Zwischenrufe

In [29]:
speech_list[1]['comments'][:5]

[{'commentator': {'name': 'Hermann Gröhe', 'party': 'CDU/CSU'},
  'text': 'Und schon wieder alles vergessen!',
  'preceding_context': 'Sehr geehrte Frau Präsidentin! Meine verehrten Kolleginnen und Kollegen! Verehrter Herr Kollege Merz, ich habe Ihnen eben sehr genau zugehört.\n('},
 {'commentator': {'name': 'Tino Chrupalla', 'party': 'AfD'},
  'text': 'Haken wir uns unter!',
  'preceding_context': 'Sehr geehrte Frau Präsidentin! Meine verehrten Kolleginnen und Kollegen! Verehrter Herr Kollege Merz, ich habe Ihnen eben sehr genau zugehört.\n(Hermann Gröhe [CDU/CSU]: Und schon wieder alles vergessen!)\nIch will Ihnen eins antworten: Unterschätzen Sie unser Land nicht! Unterschätzen Sie nicht die Bürgerinnen und Bürger dieses Landes!\n(Beifall bei der SPD, dem GRÜNE und der FDP\xa0– Lachen des Abg. Tino Chrupalla [AfD])\nIn schweren Zeiten wächst unser Land über sich selbst hinaus.\n('},
 {'commentator': {'name': 'Tino Chrupalla', 'party': 'AfD'},
  'text': 'Ja! Wir haken uns unter!',
  

... und den Beifall

In [30]:
speech_list[1]['applause']

{'CDU/CSU': 0,
 'GRÜNE': 40,
 'SPD': 40,
 'FDP': 38,
 'AfD': 1,
 'DIE LINKE': 0,
 'KPD': 0,
 'BP': 0,
 'DP': 0,
 'WAV': 0,
 'Z': 0,
 'fraktionslos': 0}

# 4. Parsen von allen Protokollen seit März 1991

In [31]:
speakers_not_found = []

In [32]:
def parse_protocol(path_to_json, path_to_output_dir):
    text = load_text(path_to_json)
    if load_text == 'no text found':
        print(f"Processed protocol {ntpath.basename(path_to_json)}. NO TEXT FOUND")
    
    count_zusammengefasst = 0 #debugging

    speeches_raw = re.split(speaker_match, text)[1:]

    speeches = []
    for i in range(0, len(speeches_raw), 5):
        if speeches_raw[i] != None and speeches_raw[i+1] != None:
            name = speeches_raw[i].strip()    
            party = speeches_raw[i+1].strip()
            if party == 'CSU' or party == 'CDU': # sometimes the the secretary tasked with writing everything down forgets to put CDU/CSU instead of CSU or CDU
                party = 'CDU/CSU'
        else:
            name = speeches_raw[i+2].replace('\n', ' ').strip() # sometimes the formatting is all over the place e.g. "Olaf\nScholz" instead of "Olaf Scholz" :)
            party = find_party_by_name(name, bt_tuples)

        speech = {
            'speaker':{
                'name':name,
                'party':party
            },
            'text': re.split(r'(\nVizepräs.{0,99}?:)|(\nPräsid.{0,99}?:)|(\nAnlage)',speeches_raw[i+4])[0].strip() # handle end of File (Anlagen) and interruptions by the Bundestagspräsident(in) or Vice Bundestagspräsident(in)
        }
    

        # info/warning about speakers which are not in the BT where their party affiliation could not be resolved
        if party not in parties:
            print(f"    Error at {speech}")
            print(f"    Capture Groups: {speeches_raw[i:i+4]}")
            print(f"    Speaker not found. Either Speaker is not part of BT,  RegEx falsely matched expression or protocol has errors. Speech will not appear in parsed version.")
            speakers_not_found.append((ntpath.basename(path_to_json), name, party))
            continue
            # This is triggered mostly by guest-speakers and name-typos. Also, sometimes a state-secretary has not been member of the BT before becoming state-secretary

        
        if len (speeches) >=1 and speeches[-1]['speaker'] == speech['speaker']:
            speeches[-1]['text'] += '\n' + speech['text'] # append interrupted speeches by same speaker e.g. after short interruptions by Bundestagspräsident(in)
            count_zusammengefasst +=1
        else:
            speeches.append(speech)

        
        comments = []
        # comments with known speaker
        for match in re.finditer(unterbrechung_match,speech['text']+'\n'): # newline for the regex to match comments right at the end of the speech
            comment = {
                'commentator': {
                    'name': match.group(1),
                    'party': match.group(2)
                },
                'text': match.group(3).strip(),
                'preceding_context': speech['text'][:match.start()] # include all text until comment for later training of LLM
            }
            comments.append(comment)
        
        # comments with unknown speaker
        for match in re.finditer(zuruf_match, re.sub('der LINKEN', 'DIE LINKE', speech['text'])):
            comment = {'commentator':{
                    'name': '<unknown>',
                    'party': match.group(1)
                },
                'text': match.group(2),
                'preceding_context': speech['text'][:match.start()]
            }
            comments.append(comment)

        speech['comments'] = comments

        # applause
        beifall = re.findall(beifall_match, speech['text'])
        beifall = ''.join(beifall)
        beifall = re.sub('der LINKEN', 'DIE LINKE', beifall)
        beifall_counts = {party: beifall.count(f' {party}') for party in parties}
        speech['applause'] = beifall_counts
  
    json_string = json.dumps(speeches, indent=4)
    print(f"Processed protocol {ntpath.basename(path_to_json)}. #speeches = {len(speeches)} #speeches_concatenated = {count_zusammengefasst}")
    # save to file
    with open(f"{path_to_output_dir}{ntpath.basename(path_to_json)}", "w") as json_file:
        json_file.write(json_string)


In [32]:
data_path = r'../_data/protocols/'
output_path = r'../_data/parsed_protocols/'

In [33]:
parse_protocol(data_path + '20_012_2022-01-14.json', output_path)

Processed protocol 20_012_2022-01-14.json. #speeches = 74 #speeches_concatenated = 27


In [34]:
all_files = os.listdir(data_path)
json_files = [filename for filename in all_files if filename.endswith('.json')]
print(f"Gesamtanzahl der Protokolle: {len(all_files)}")
c = 0
for filename in all_files:
    parse_protocol(data_path + filename, output_path)
    c+=1
    if c % 20 == 0:
        print(f"{(c / len(all_files)) * 100:.2f}% der Protokolle verarbeitet")

Gesamtanzahl der Protokolle: 2071
Processed protocol 12_013_1991-03-12.json. #speeches = 72 #speeches_concatenated = 14
Processed protocol 12_014_1991-03-13.json. #speeches = 90 #speeches_concatenated = 28
Processed protocol 12_015_1991-03-14.json. #speeches = 96 #speeches_concatenated = 31
Processed protocol 12_016_1991-03-15.json. #speeches = 26 #speeches_concatenated = 4
Processed protocol 12_017_1991-03-20.json. #speeches = 76 #speeches_concatenated = 15
    Error at {'speaker': {'name': 'Wolfang Gröbl', 'party': '<unknown>'}, 'text': 'Im Gegenteil, \ndie Berücksichtigung der ökologischen Probleme \nführt zu dieser jetzt gewählten Va riante, im übrigen \nauch im Hinblick auf die Verbindung Straßburg-Kehl, \ndie zeitlich als Konkurrenzstrecke zu der von Ihnen \nnachgefragten Strecke gesehen werden muß.'}
    Capture Groups: [None, None, 'Wolfang Gröbl', 'Parl. Staatssekretär']
    Speaker not found. Either Speaker is not part of BT,  RegEx falsely matched expression or protocol has 

In [35]:
import csv

with open('../_data/not_part_of_bt.csv','w', encoding='utf8', newline='') as out:
    csv_out=csv.writer(out)
    csv_out.writerows(speakers_not_found) 

# 5. Vorbereiten für die statistische Analyse

Sätze der Reden zählen und relevante Daten ein Dataframe laden, das per csv abgespeichert werden kann.  
So wird das Laden und die Handhabung der für die Datenanalyse relevanten Unterbrechungen schneller, weil nur eine Dateioperation nötig ist und für die Analyse unwichtige Felder, wie der vorangegangene Redetext wegfallen.

In [36]:
from tqdm import tqdm
from spacy.lang.de import German
import pandas as pd
import os
import json

In [37]:
relevant_protocols = os.listdir('../_data/parsed_protocols/' )
relevant_protocols = [file for file in relevant_protocols if file.endswith('.json')]
nlp = German()
nlp.add_pipe('sentencizer')
speech_id =0

comment_list = []
speech_list = []
for protocol_filename in tqdm(relevant_protocols):
    with open('../_data/parsed_protocols/'+protocol_filename) as f:
        protocol = json.load(f)
        for speech in protocol:
            speech_id +=1
            speaker_name = speech['speaker']['name']
            speaker_party = speech['speaker']['party']
            doc = nlp(speech['text'])
            speech_len_sents = sum(1 for sent in doc.sents)

            speech_list.append([
                speaker_party,
                speaker_name,
                speech['applause'],
                speech_len_sents,
                protocol_filename[7:-5],
                speech_id
            ])

            if 'comments' in speech:
                for comment in speech['comments']:
                    comment_list.append([comment['text'],
                                  comment['commentator']['party'],
                                  comment['commentator']['name'],
                                  speaker_party,
                                  speaker_name,
                                  protocol_filename[7:-5],
                                  speech_len_sents, 
                                  speech_id
                                ])


100%|██████████| 2071/2071 [09:19<00:00,  3.70it/s]


In [38]:
print(len(comment_list))

477702


In [39]:
print(len(speech_list))

229372


In [40]:
columns_comments=['comment_text', 'comment_party', 'comment_name',  'interrupted_speaker_party','interrupted_speaker', 'date', 'speech_len_sents', 'speech_id']
df_interruptions = pd.DataFrame(comment_list, columns = columns_comments)

columns_speeches=['speaker_party', 'speaker', 'applause', 'speech_len_sents', 'date','speech_id']
df_speeches = pd.DataFrame(speech_list, columns=columns_speeches)
# denormalized "databases" because frankly it does not matter for the amount of doubled data. On top of that, the data in itself is static and not susceptible to change so it only 'costs' some Megabytes of storage

In [41]:
df_interruptions.head()

,comment_text,comment_party,comment_name,interrupted_speaker_party,interrupted_speaker,date,speech_len_sents,speech_id
0,So kriegen Sie \ndie Steuerlüge nicht aus der ...,DIE LINKE,Dr. B riefs,CDU/CSU,Dr. Theodor Waigel,1991-03-12,386,1
1,Bravo!,SPD,<unknown>,CDU/CSU,Dr. Theodor Waigel,1991-03-12,386,1
2,Das ist eine \nFrechheit!,DIE LINKE,<unknown>,CDU/CSU,Dr. Jürgen Rüttgers,1991-03-12,15,3
3,Gebt erst \nmal das Vermögen der DIE LINKE dem...,CDU/CSU,zu Bentrup,DIE LINKE,Dr. Ulrich Briefs,1991-03-12,127,7
4,Die DIE LINKE \nsoll das Vermögen zurückgeben!,CDU/CSU,zu Bentrup,DIE LINKE,Dr. Ulrich Briefs,1991-03-12,127,7
...,...,...,...,...,...,...,...,...
477697,Und ihr schimpft immer auf uns!,CDU/CSU,Dr. Anja Weisgerber,SPD,Mathias Stein,2024-06-14,41,229372
477698,Wohlfeil war das!,GRÜNE,Stefan Gelbhaar,SPD,Mathias Stein,2024-06-14,41,229372
477699,Es war ein Kontrollversagen! Ich habe nicht au...,CDU/CSU,Dr. Anja Weisgerber,SPD,Mathias Stein,2024-06-14,41,229372
477700,Kontrollversagen!,CDU/CSU,Dr. Anja Weisgerber,SPD,Mathias Stein,2024-06-14,41,229372


Leider sind noch Zeilenumbrüche enthalten...

In [42]:
def clean_double_whitespace(text):
    cleaned_text = ' '.join(text.split())
    cleaned_text = cleaned_text.strip()
    return cleaned_text

### Säubern und Abspeichern der Daten

Überprüfen, ob es fehlende Werte gibt:

In [43]:
rows_with_na = df_interruptions[df_interruptions.isna().any(axis=1)]
rows_with_na

In [44]:
df_interruptions['comment_text'] = df_interruptions['comment_text'].str.replace('\n', ' ')
df_interruptions['comment_name'] = df_interruptions['comment_name'].str.replace('\n', ' ')
df_interruptions['interrupted_speaker'] = df_interruptions['interrupted_speaker'].str.replace('\n', ' ')

df_interruptions['comment_text'] = df_interruptions['comment_text'].apply(clean_double_whitespace)
df_interruptions['comment_name'] = df_interruptions['comment_name'].apply(clean_double_whitespace)
df_interruptions['interrupted_speaker'] = df_interruptions['interrupted_speaker'].apply(clean_double_whitespace)

Leider schreibt der Protokollant/die Protokollantin nicht immer einheitlich mit und schreibt statt "CDU/CSU" einfach eine der beiden Parteien in das Protokoll.

In [45]:
df_interruptions.loc[df_interruptions['comment_party'].isin(['CDU', 'CSU']), 'comment_party'] = 'CDU/CSU'

df_interruptions.loc[df_interruptions['interrupted_speaker_party'].isin(['CDU', 'CSU']), 'interrupted_speaker_party'] = 'CDU/CSU'

In [46]:
df_interruptions.to_csv('../_data/interruptions.csv') # Abspeichern

In [47]:
df_speeches['speaker'] = df_speeches['speaker'].str.replace('\n', ' ')
df_speeches['speaker'] = df_speeches['speaker'].apply(clean_double_whitespace)
df_speeches['applause'] = df_speeches['applause'].apply(json.dumps)

In [48]:
df_speeches.to_csv('../_data/speeches.csv')

# Preparing Data for labeling

In [2]:
relevant_protocols = os.listdir('../_data/parsed_protocols/' )
relevant_protocols = [file for file in relevant_protocols if file.endswith('.json')]
nlp = German()
nlp.add_pipe('sentencizer')
speech_id =0

comment_list = []
speech_list = []
for protocol_filename in tqdm(relevant_protocols):
    with open('../_data/parsed_protocols/'+protocol_filename) as f:
        protocol = json.load(f)
        for speech in protocol:
            speech_id +=1
            speaker_name = speech['speaker']['name']
            speaker_party = speech['speaker']['party']
            #doc = nlp(speech['text'])
            #speech_len_sents = sum(1 for sent in doc.sents)

            speech_list.append([
                speaker_party,
                speaker_name,
                speech['applause'],
                #speech_len_sents,
                protocol_filename[7:-5],
                speech_id
            ])

            if 'comments' in speech:
                for comment in speech['comments']:
                    comment_list.append([comment['text'],
                                  comment['commentator']['party'],
                                  comment['commentator']['name'],
                                  speaker_party,
                                  speaker_name,
                                  protocol_filename[7:-5],
                                  #speech_len_sents, 
                                  speech_id
                                ])


100%|██████████| 2064/2064 [00:22<00:00, 92.84it/s] 


In [3]:
columns_comments=['comment_text', 'comment_party', 'comment_name',  'interrupted_speaker_party','interrupted_speaker', 'date', 'speech_id']
df_comments = pd.DataFrame(comment_list, columns = columns_comments)

columns_speeches=['speaker_party', 'speaker', 'applause',  'date','speech_id']
df_speeches = pd.DataFrame(speech_list, columns=columns_speeches)
# denormalized "databases" because frankly it does not matter for the amount of doubled data. On top of that, the data in itself is static and not susceptible to change so it only 'costs' some Megabytes of storage

In [75]:
def clean_double_whitespace(text):
    cleaned_text = ' '.join(text.split())
    cleaned_text = cleaned_text.strip()
    return cleaned_text

In [76]:
df_comments['comment_text'] = df_comments['comment_text'].str.replace('\n', ' ')
df_comments['comment_name'] = df_comments['comment_name'].str.replace('\n', ' ')
df_comments['interrupted_speaker'] = df_comments['interrupted_speaker'].str.replace('\n', ' ')

df_comments['comment_text'] = df_comments['comment_text'].apply(clean_double_whitespace)
df_comments['comment_name'] = df_comments['comment_name'].apply(clean_double_whitespace)
df_comments['interrupted_speaker'] = df_comments['interrupted_speaker'].apply(clean_double_whitespace)

df_comments.loc[df_comments['comment_party'].isin(['CDU', 'CSU']), 'comment_party'] = 'CDU/CSU'

df_comments.loc[df_comments['interrupted_speaker_party'].isin(['CDU', 'CSU']), 'interrupted_speaker_party'] = 'CDU/CSU'

In [6]:
df_speeches['speaker'] = df_speeches['speaker'].str.replace('\n', ' ')
df_speeches['speaker'] = df_speeches['speaker'].apply(clean_double_whitespace)
df_speeches['applause'] = df_speeches['applause'].apply(json.dumps)

In [51]:
len(df_speeches)

228165

# Debugging

In [99]:
data_path = '../_data/protocols/12_013_1991-03-12.json'
data_path = '../_data/protocols/12_015_1991-03-14.json'
data_path = '../_data/protocols/12_014_1991-03-13.json'
data_path = '../_data/protocols/14_215_2002-01-31.json'
data_path = '../_data/protocols/15_026_2003-02-14.json'
parse_protocol(data_path, '../_data/example_txt_protocols_testing/')

Processed protocol 15_026_2003-02-14.json. #speeches = 44 #speeches_concatenated = 13


In [100]:
text = load_text(data_path)

with open('../_data/example_txt_protocols_testing/' + ntpath.basename(data_path)[:-4]+ 'txt', 'w', encoding = 'utf8') as f:
        f.write(text)

In [54]:
def clean_double_whitespace(text):
    if type(text) == float:
        return None
    cleaned_text = ' '.join(text.split())
    cleaned_text = cleaned_text.strip()
    return cleaned_text

df_comments_old = pd.read_csv('../_data/interruptions.csv')
df_comments_old = df_comments_old[df_comments_old['date'] >='1991-03-13']
df_comments_old['comment_text'] = df_comments_old['comment_text'].apply(clean_double_whitespace)
df_comments_old['comment_name'] = df_comments_old['comment_name'].apply(clean_double_whitespace)
df_comments_old['interrupted_speaker'] = df_comments_old['interrupted_speaker'].apply(clean_double_whitespace)


df_comments_old

,Unnamed: 0,comment_text,comment_party,comment_name,interrupted_speaker_party,interrupted_speaker,date,speech_len_sents,speech_id
90,90,Das ist der Herr Missionar!,SPD,des Abg. Duve,SPD,Dr. Hans-Jochen Vogel,1991-03-13,395,98
91,91,Gestern bei Herrn Lafontaine klang das aber an...,FDP,Dr. Graf Lambsdorff,SPD,Dr. Hans-Jochen Vogel,1991-03-13,395,98
92,92,Er glaubt sich selber nicht!,SPD,<unknown>,SPD,Dr. Hans-Jochen Vogel,1991-03-13,395,98
93,93,Der ist gar nicht da!,CDU/CSU,<unknown>,SPD,Dr. Hans-Jochen Vogel,1991-03-13,395,98
94,94,Nur Modrow!,CDU/CSU,<unknown>,SPD,Dr. Hans-Jochen Vogel,1991-03-13,395,98
...,...,...,...,...,...,...,...,...,...
461178,461178,Doch! In XBau 2.4!,FDP,Daniel Föst,CDU/CSU,Michael Kießling,2024-05-16,42,203046
461179,461179,Warum immer noch?,FDP,Daniel Föst,CDU/CSU,Michael Kießling,2024-05-16,42,203046
461180,461180,Sie haben das vergessen gehabt mit der Digital...,GRÜNE,Christina-Johanne Schröder,CDU/CSU,Michael Kießling,2024-05-16,42,203046
461181,461181,"Würden Ihre Länder digitalisieren, müsste es k...",FDP,Daniel Föst,CDU/CSU,Michael Kießling,2024-05-16,42,203046


In [72]:
df_comments_old['is_float'] = df_comments_old['comment_text'].apply(lambda x: isinstance(x, float))
df_comments_old[df_comments_old['is_float']]

,Unnamed: 0,comment_text,comment_party,comment_name,interrupted_speaker_party,interrupted_speaker,date,speech_len_sents,speech_id,is_float


In [66]:
df_comments[((df_comments['date'] == '1991-03-14') &(df_comments['interrupted_speaker'] == 'Harald B. Schäfer'))]

,comment_text,comment_party,comment_name,interrupted_speaker_party,interrupted_speaker,date,speech_id
70,Das tun wir!,CDU/CSU,<unknown>,SPD,Harald B. Schäfer,1991-03-14,91
71,Ihm das erst vor-werfen und dann selbst nichts...,CDU/CSU,<unknown>,SPD,Harald B. Schäfer,1991-03-14,91


In [127]:
df_comments[((df_comments['comment_text'] == 'Skandal!') & (df_comments['date'] == '1991-03-14'))]

,comment_text,comment_party,comment_name,interrupted_speaker_party,interrupted_speaker,date,speech_id
92,Skandal!,SPD,<unknown>,SPD,Ottmar Schreiner,1991-03-14,119


In [14]:
df_comments[((df_comments['comment_text'] == 'Aus Erfahrung!') & (df_comments['date'] == '1991-09-04'))]

,comment_text,comment_party,comment_name,interrupted_speaker_party,interrupted_speaker,date,speech_id
3233,Aus Erfahrung!,SPD,Freimut Duve,CDU/CSU,Dr. Helmut Kohl,1991-09-04,2538


In [16]:
df_comments[((df_comments['comment_text'] == 'Das ist falsch!') & (df_comments['date'] == '1991-09-20'))]

,comment_text,comment_party,comment_name,interrupted_speaker_party,interrupted_speaker,date,speech_id
4149,Das ist falsch!,CDU/CSU,Dr. Paul Hoffacker,SPD,Dr. Martin Pfaff,1991-09-20,2949


In [52]:
df_comments[((df_comments['comment_text'] == 'Wir haben ja 5 Millionen Bürgergeldempfänger!'))]

,comment_text,comment_party,comment_name,interrupted_speaker_party,interrupted_speaker,date,speech_id
456210,Wir haben ja 5 Millionen Bürgergeldempfänger!,CDU/CSU,Klaus-Peter Willsch,SPD,Fabian Funke,2024-05-16,228060


In [98]:
df_comments[((df_comments['comment_name'].str.contains('Margit Spielmann')))]

,comment_text,comment_party,comment_name,interrupted_speaker_party,interrupted_speaker,date,speech_id


In [92]:
comment_names_old = set(df_comments_old['comment_name'])
comment_names_new = set(df_comments['comment_name'])

# Finding comment_names that are in df_comments_new but not in df_comments
unique_comment_names = comment_names_old - comment_names_new

# Converting the result to a DataFrame for better display
unique_comment_names_df = pd.DataFrame(list(unique_comment_names), columns=['comment_name'])
unique_comment_names_df[:60]

,comment_name
0,Dr. Wer-ner Hoyer
1,Birgitt BenderAustermann
2,15948 Michelbach
3,Marie Steen
4,Dr. Fritz Gautier
5,von Schönburg-Glauchau
6,Andreas Schockenhoff
7,Dietrich Auster-mann
8,von Polheim
9,Jürgen KoppelinWesterwelle


In [77]:
merged_df = pd.merge(df_comments, df_comments_old, on=['date', 'comment_party', 'comment_name'], how='outer', indicator=True)

# Filtering to get the rows that are only in df_comments_new
unique_to_new = merged_df[merged_df['_merge'] == 'left_only'].drop(columns=['_merge'])

# Display the result
unique_to_new

,comment_text_x,comment_party,comment_name,interrupted_speaker_party_x,interrupted_speaker_x,date,speech_id_x,Unnamed: 0,comment_text_y,interrupted_speaker_party_y,interrupted_speaker_y,speech_len_sents,speech_id_y
2722,Der Meinung sind wir auch!,DIE LINKE,Dr. B riefs,CDU/CSU,Wilhelm Rawe,1991-03-14,113.0,NaN,NaN,NaN,NaN,NaN,NaN
3377,Sehr wahr!,CDU/CSU,Freiherr von Schorlemer,SPD,Gernot Erler,1991-03-15,206.0,NaN,NaN,NaN,NaN,NaN,NaN
3402,Der ist perspektivlos!,SPD,<unknown>,CDU/CSU,Dr. Ottfried Hennig,1991-03-20,235.0,NaN,NaN,NaN,NaN,NaN,NaN
4382,Was hat das mit dem Gesetz zu tun?,CDU/CSU,Dr. Freiherr von Stetten,GRÜNE,Gerd Poppe,1991-03-22,490.0,NaN,NaN,NaN,NaN,NaN,NaN
4383,Das machen wir nicht!,CDU/CSU,Dr. Freiherr von Stetten,GRÜNE,Gerd Poppe,1991-03-22,490.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5550234,Halt die Klappe!,AfD,Beatrix von Storch,AfD,Dr. Christina Baum,2024-05-16,228012.0,NaN,NaN,NaN,NaN,NaN,NaN
5550235,Halt die Klappe!,AfD,Beatrix von Storch,AfD,Dr. Christina Baum,2024-05-16,228012.0,NaN,NaN,NaN,NaN,NaN,NaN
5550236,… 1 Million einzusperren!,AfD,Beatrix von Storch,CDU/CSU,Stephan Pilsinger,2024-05-16,228016.0,NaN,NaN,NaN,NaN,NaN,NaN
5550237,Ich sperre 20 Millionen Menschen ein! Oder 80 ...,AfD,Beatrix von Storch,CDU/CSU,Stephan Pilsinger,2024-05-16,228016.0,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
subset_columns = ['date','comment_party', 'comment_name']
df_old_subset = df_comments_old[subset_columns]
df_new_subset = df_comments[subset_columns]

# Perform an anti-join to find rows in df_comments_old that are not in df_comments
merged = df_old_subset.merge(df_new_subset, on=subset_columns, how='left', indicator=True)
df_not_in_new = df_comments_old[merged['_merge'] == 'left_only']

df_not_in_new

C:\Users\Robin\AppData\Local\Temp\ipykernel_3304\3774583327.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_not_in_new = df_comments_old[merged['_merge'] == 'left_only']


,Unnamed: 0,comment_text,comment_party,comment_name,interrupted_speaker_party,interrupted_speaker,date,speech_len_sents,speech_id,is_float
3396,3396,Aus Erfahrung!,SPD,Freimut Duve,CDU/CSU,Dr. Helmut Kohl,1991-09-04,396,2198,False
3682,3682,"Wo liegen die denn, Herr Kollege Kirschner?",CDU/CSU,<unknown>,SPD,Klaus Kirschner,1991-09-05,146,2254,False
4323,4323,Das ist falsch!,CDU/CSU,Dr. Paul Hoffacker,SPD,Dr. Martin Pfaff,1991-09-20,64,2602,False
4324,4324,Das ist was Neues!,CDU/CSU,Dr. Paul Hoffacker,SPD,Dr. Martin Pfaff,1991-09-20,64,2602,False
4342,4342,"Doch, die Ge",CDU/CSU,Friedrich Bohl,FDP,Cornelia Schmalz-Jacobsen,1991-09-25,73,2675,False
...,...,...,...,...,...,...,...,...,...,...
460079,460079,"Das ist doch ein Potemkinsches Dorf, das Sie h...",CDU/CSU,Thorsten Frei,GRÜNE,Felix Banaszak,2024-04-26,63,202670,False
460080,460080,Machen!,CDU/CSU,<unknown>,FDP,Manfred Todtenhausen,2024-04-26,36,202671,False
461091,461091,Dank euch!,AfD,Tino Chrupalla,SPD,Fabian Funke,2024-05-16,51,203015,False
461092,461092,Wir haben ja 5 Millionen Bürgergeldempfänger!,CDU/CSU,Klaus-Peter Willsch,SPD,Fabian Funke,2024-05-16,51,203015,False


In [133]:
print(len(df_not_in_new))

40600


In [5]:

from tqdm import tqdm


relevant_protocols = [filepath for filepath in os.listdir('../_data/parsed_protocols/') if filepath.endswith('.json')]
speech_id =0

comment_list = []
for protocol_filename in tqdm(relevant_protocols):
    with open('../_data/parsed_protocols/'+protocol_filename) as f:
        protocol = json.load(f)
        for speech in protocol:
            speech_id +=1
            speaker_name = speech['speaker']['name']
            speaker_party = speech['speaker']['party']
            if 'comments' in speech:
                for comment in speech['comments']:
                    if(len(comment['preceding_context'])< 300):
                        context_len = len(comment['preceding_context'])
                    else:
                        context_len=300
                    comment_list.append([comment['text'],
                                  comment['commentator']['party'],
                                  comment['commentator']['name'],
                                  speaker_party,
                                  speaker_name,
                                  protocol_filename[7:-5],
                                  speech_id, 
                                  comment['preceding_context'][-context_len:].replace('\n', ' ')
                                ])
columns=['comment_text', 'comment_party', 'comment_name',  'interrupted_speaker_party','interrupted_speaker', 'date','speech_id', 'preceding_context']
df_labeling = pd.DataFrame(comment_list, columns = columns)

  0%|          | 0/2075 [00:00<?, ?it/s]

100%|██████████| 2075/2075 [00:32<00:00, 64.69it/s]


In [6]:
df_labeling['comment_text'] = df_labeling['comment_text'].str.replace('\n', ' ')
df_labeling['comment_name'] = df_labeling['comment_name'].str.replace('\n', ' ')
df_labeling['interrupted_speaker'] = df_labeling['interrupted_speaker'].str.replace('\n', ' ')

In [7]:
df_labeling.loc[df_labeling['comment_party'].isin(['CDU', 'CSU']), 'comment_party'] = 'CDU/CSU'
df_labeling.loc[df_labeling['interrupted_speaker_party'].isin(['CDU', 'CSU']), 'interrupted_speaker_party'] = 'CDU/CSU'

In [8]:
df_labeling = df_labeling.sample(3000, random_state=42)
df_labeling.iloc[:1000].to_csv('./_data/comments_for_labeling_linh.csv') # Abspeichern
df_labeling.iloc[1000:2000].to_csv('./_data/comments_for_labeling_eddi.csv') # Abspeichern
df_labeling.iloc[2000:3000].to_csv('./_data/comments_for_labeling_robin.csv') # Abspeichern

In [33]:
data_path = r'..\_data\protocols\12_004_1991-01-18.json'
data_path = r'..\_data\protocols\16_092_2007-03-30.json'
data_path = r'..\_data\protocols\20_114_2023-07-05.json'
data_path = r'..\_data\protocols\20_115_2023-07-06.json'
data_path = r'..\_data\protocols\13_182_1997-06-13.json'
data_path = r'..\_data\protocols\14_099_2000-04-13.json'
data_path = r'..\_data\protocols\20_169_2024-05-16.json'
data_path= r'..\_data\protocols\12_113_1992-10-15.json'
data_path = '../_data/protocols/16_066_2006-11-22.json'
output_path = '../_data/example_txt_protocols_testing/'
parse_protocol(data_path, output_path)

Processed protocol 16_066_2006-11-22.json. #speeches = 70 #speeches_concatenated = 35


In [35]:
test_text = load_text(data_path)
with open('../_data/example_txt_protocols_testing/' + ntpath.basename(data_path)[:-4]+ 'txt', 'w', encoding = 'utf8') as f:
        f.write(test_text)

In [63]:
buendnis_match = re.compile(r'(BÜND(-\n?)?NIS((-\n?)?SES)?\s*?90\/\s*(DIE\s*?)?GRÜ(-\n?)?NEN?)', flags=re.UNICODE|re.IGNORECASE)

In [73]:
text = load_text(data_path)
print(f"loaded text from {data_path}")
print(re.findall(buendnis_match, text))
text = re.sub(buendnis_match, 'GRÜNE', text)
print(re.findall(buendnis_match, text))

with open('./data/test/' + ntpath.basename(data_path)[:-4]+ 'txt', 'w', encoding = 'utf8') as f:
        f.write(text)

loaded text from .\data\opendata_api\protocols\12_113_1992-10-15.json
[]
[]


In [106]:
datapath = r'data\opendata_api\protocols\14_099_2000-04-13.json'

with open(datapath, 'r', encoding = 'utf8') as f:
        text = json.load(f)['text']

with open('./data/examples/' + ntpath.basename(data_path)[:-4]+ 'txt', 'w', encoding = 'utf8') as f:
        f.write(text)

In [132]:
speakers_not_found = [snf[:2] for snf in speakers_not_found]
print(speakers_not_found[1])

('12_023_1991-04-25.json', 'Rainer Runke')


In [134]:
import csv

with open('./data/speakers_not_part_of_bt.csv','w', encoding='utf8', newline='') as out:
    csv_out=csv.writer(out)
    csv_out.writerows(speakers_not_found) 


In [136]:
parsed_path = './data/opendata_api/parsed_protocols/'
n_speeches = 0
for filename in os.listdir(parsed_path):
    with open(parsed_path+filename) as f:
        protocol = json.load(f)
    n_speeches += len(protocol)
    

In [137]:
n_speeches

223500

In [138]:
len(speakers_not_found)/n_speeches

0.001087248322147651

# Sonderfälle

## Fehler in Protokollen:

###data\test\14_099_2000-04-13.txt Zeile ab 2503

manche Redner wie z.B. amtierende Minister, Staatssekretäre etc. fallen aus dem Raster heraus und es steht keine Partei dahinter -> Wir wollen trotzdem die Parteien anmerken  
-> Abgleich mit XML-Stammdatenliste. Hier besonders angenehm: Schema: <p>"&lt;Vorname> &lt;Name>, &lt;Titel>:" </p> -> Das Ganze auch OHNE Doktortitel bei z.B. Dr. Robert Habeck -> Auslesen von Stammdaten xml

- Beatrix von Storch als Sonderfall

Sonderfall Einwürfe bei 1951-Protokollen. und 1971 "(Abg. <nachname>: <text>)
1951: Auch neuer Textanfang: (Dr.)? <nachname> (<partei>) (<Wahlkreis>)? :
1971 + 1981: (Dr.)? <nachname> (<partei>) (<Wahlkreis>)?: